## Step 1

In [1]:
# Install the Google ADK
!pip install google-adk -q

## Step 2

In [11]:
import os
import vertexai
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import AgentTool, google_search
from google.genai import types

# --- 1. CONFIGURATION ---
PROJECT_ID = "qwiklabs-gcp-04-cc20aba4b1cc"
LOCATION = "us-central1"

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
vertexai.init(project=PROJECT_ID, location=LOCATION)

# --- 2. DEFINE THE SPECIALIZED AGENTS ---

greeter = Agent(
    name="Greeter",
    instruction="Greet the user and explain the Answer Team is starting their research.",
    model="gemini-2.0-flash"
)

searcher = Agent(
    name="Searcher",
    instruction="Use Google Search to find factual details about the user's question.",
    tools=[google_search],
    model="gemini-2.0-flash"
)

critique = Agent(
    name="Critique",
    instruction="Review search results. Suggest specific improvements for clarity and accuracy.",
    model="gemini-2.0-flash"
)

refiner = Agent(
    name="Refiner",
    instruction="Rewrite the final answer based on the search data and the critique suggestions.",
    model="gemini-2.0-flash"
)

# --- 3. THE WORKFLOW MANAGER (Orchestrator) ---
# We use AgentTool to wrap them so they are accepted in the 'tools' list.
workflow_orchestrator = Agent(
    name="WorkflowManager",
    instruction="""You are the Workflow Manager. You MUST follow this EXACT sequence:
    1. Call the Greeter tool to welcome the user.
    2. Call the Searcher tool to get the facts.
    3. Call the Critique tool to evaluate those facts.
    4. Call the Refiner tool to produce the final verified answer.
    Provide the Refiner's output as the final response.""",
    tools=[
        AgentTool(agent=greeter),
        AgentTool(agent=searcher),
        AgentTool(agent=critique),
        AgentTool(agent=refiner)
    ],
    model="gemini-2.0-flash"
)

# --- 4. RUNNER SETUP ---
runner = Runner(
    app_name="WorkflowApp",
    agent=workflow_orchestrator,
    session_service=InMemorySessionService()
)

# --- 5. EXECUTION & OUTPUT ---
async def run_challenge_four():
    query = "What is the current status of the men's NCAA basketball rankings?"
    print(f"--- Challenge Four: Workflow Test ---\nQuery: {query}\n")

    session = await runner.session_service.create_session(app_name="WorkflowApp", user_id="user_workflow")

    async for event in runner.run_async(
        session_id=session.id,
        user_id="user_workflow",
        new_message=types.Content(role="user", parts=[types.Part(text=query)])
    ):
        # IMPROVED EVENT TRACKING:
        # Check if the event contains a function call (which is how AgentTools are triggered)
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.function_call:
                    # This will print: [DELEGATION]: Calling agent/tool -> Greeter
                    print(f"[DELEGATION]: Calling agent/tool -> {part.function_call.name}")

        is_final = event.is_final_response() if callable(event.is_final_response) else event.is_final_response
        if is_final:
            print(f"\n--- FINAL REFINED OUTPUT ---\n{event.content.parts[0].text}")

# Run the test
await run_challenge_four()

--- Challenge Four: Workflow Test ---
Query: What is the current status of the men's NCAA basketball rankings?

[DELEGATION]: Calling agent/tool -> Greeter
[DELEGATION]: Calling agent/tool -> Searcher
[DELEGATION]: Calling agent/tool -> Critique
[DELEGATION]: Calling agent/tool -> Refiner

--- FINAL REFINED OUTPUT ---
The current status of the men's NCAA basketball rankings as of early March 2026 are as follows:

**TSN Power Rankings (March 5, 2026):**

1. Duke Blue Devils (28-2)
2. Arizona Wildcats (28-2)
3. Michigan Wolverines (27-2)
4. UConn Huskies (27-3)
5. Florida Gators (24-6)
6. Houston Cougars (25-5)
7. Michigan State Spartans (24-5)
8. Illinois Fighting Illini (23-7)
9. Iowa State Cyclones (24-6)
10. Purdue Boilermakers (23-7)
11. Nebraska Cornhuskers (25-5)
12. Gonzaga Bulldogs (28-3)
13. Texas Tech Red Raiders (22-8)
14. Kansas Jayhawks (21-9)
15. Alabama Crimson Tide (22-8)
16. Virginia Cavaliers (26-4)
17. Arkansas Razorbacks (22-8)
18. North Carolina Tar Heels (24-6)
19.